## Problem Formulation and Data Ingestion

The objective of this project is to evaluate the effectiveness of sequential modeling approaches on real-world sentiment classification using the "To Vaccinate or Not to Vaccinate" dataset. The task requires categorizing social media posts into positive, neutral, or negative attitudes toward immunization.

To ensure empirical validity across all comparative experiments, this section of the notebook handles data loading, exploratory analysis, cleaning, and static data partitioning. Generating fixed splits at this stage guarantees that all baseline, recurrent, convolutional, and transformer models evaluate against an identical test set.

In [1]:
import pandas as pd
import numpy as np

# Defining the file path from the Kaggle input directory
file_path = '/kaggle/input/datasets/arsenekabasinga/to-vaccinate-or-not-to-vaccinate/Train.csv'

# Loading the raw dataset
df = pd.read_csv(file_path)

# Displaying dataset dimensions and initial structure
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 Records:")
display(df.head())

# Inspecting column data types and missing values
print("\nMissing Values Count:")
print(df.isnull().sum())

print("\nRaw Label Distribution:")
print(df['label'].value_counts(dropna=False))

Dataset Shape: (10001, 4)

First 5 Records:


,tweet_id,safe_text,label,agreement
0,CL1KWCMY,Me &amp; The Big Homie meanboy3000 #MEANBOY #M...,0.0,1.0
1,E3303EME,I'm 100% thinking of devoting my career to pro...,1.0,1.0
2,M4IVFSMS,"#whatcausesautism VACCINES, DO NOT VACCINATE Y...",-1.0,1.0
3,1DR6ROZ4,I mean if they immunize my kid with something ...,-1.0,1.0
4,J77ENIIE,Thanks to <user> Catch me performing at La Nui...,0.0,1.0



Missing Values Count:
tweet_id     0
safe_text    0
label        1
agreement    2
dtype: int64

Raw Label Distribution:
label
 0.000000    4908
 1.000000    4053
-1.000000    1038
 NaN            1
 0.666667       1
Name: count, dtype: int64


### Data Cleaning and Label Standardization

Exploratory inspection of the raw records reveals two primary data quality anomalies:
1. One record contains a missing target label (`NaN`).
2. One record contains an invalid non-discrete label (`0.666667`).

Because valid classification requires discrete target classes (-1 for Negative, 0 for Neutral, and 1 for Positive), anomalous and null records are pruned. 

Additionally, deep learning frameworks (such as PyTorch and TensorFlow) require categorical cross-entropy targets to be non-negative integers indexed from $0$ to $C-1$. Consequently, the labels are mapped as follows:
* `-1` (Negative) $\rightarrow$ `0`
* `0` (Neutral) $\rightarrow$ `1`
* `1` (Positive) $\rightarrow$ `2`

In [2]:
# Removing records with missing text or target labels
df_clean = df.dropna(subset=['safe_text', 'label']).copy()

# Retaining only valid sentiment labels: -1.0, 0.0, and 1.0
df_clean = df_clean[df_clean['label'].isin([-1.0, 0.0, 1.0])].copy()

# Casting text to string type to prevent downstream tokenizer failures
df_clean['safe_text'] = df_clean['safe_text'].astype(str)

# Mapping labels to 0, 1, 2 for cross-entropy compatibility
label_mapping = {-1.0: 0, 0.0: 1, 1.0: 2}
df_clean['label_idx'] = df_clean['label'].map(label_mapping).astype(int)

# Displaying post-cleaning distribution
print(f"Cleaned Dataset Shape: {df_clean.shape}")
print("\nStandardized Class Distribution (0=Negative, 1=Neutral, 2=Positive):")
print(df_clean['label_idx'].value_counts().sort_index())

Cleaned Dataset Shape: (9999, 5)

Standardized Class Distribution (0=Negative, 1=Neutral, 2=Positive):
label_idx
0    1038
1    4908
2    4053
Name: count, dtype: int64


### Stratified Data Partitioning

The cleaned records are partitioned into three independent subsets:
* **Training Set (70%):** Utilized for model parameter estimation and representation learning.
* **Validation Set (15%):** Utilized for hyperparameter tuning, regularization selection, and early stopping.
* **Test Set (15%):** Held out strictly for final metric reporting and cross-model comparison.

Stratified sampling is enforced using a fixed pseudo-random seed (`random_state=42`). Stratification guarantees that the proportional representation of each sentiment class remains uniform across all splits, preventing distribution shift between training and evaluation.

In [4]:
from sklearn.model_selection import train_test_split

# Initial partition: 85% train-validation combined, 15% holdout test
train_val_df, test_df = train_test_split(
    df_clean,
    test_size=0.15,
    random_state=42,
    stratify=df_clean['label_idx']
)

# Secondary partition: Split the 85% into 70% train and 15% validation
# (0.17647 of 85% is approximately 15% of the total dataset)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.17647,
    random_state=42,
    stratify=train_val_df['label_idx']
)

# Output partition dimensions
print(f"Training Instances:   {len(train_df)} ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"Validation Instances: {len(val_df)} ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f"Test Instances:       {len(test_df)} ({len(test_df)/len(df_clean)*100:.1f}%)")

Training Instances:   6999 (70.0%)
Validation Instances: 1500 (15.0%)
Test Instances:       1500 (15.0%)


### Verification and Artifact Export

To verify the integrity of the stratified partitioning, class proportions are compared across the three subsets. Once verified, the data subsets are saved as static CSV files to the working directory. These files serve as the single source of truth across all subsequent modeling notebooks.

In [5]:
# Computing and verifying class proportions across all subsets
verification_df = pd.DataFrame({
    'Train (%)': train_df['label_idx'].value_counts(normalize=True).sort_index() * 100,
    'Validation (%)': val_df['label_idx'].value_counts(normalize=True).sort_index() * 100,
    'Test (%)': test_df['label_idx'].value_counts(normalize=True).sort_index() * 100
})
print("Class Distribution Comparison Across Splits:")
display(verification_df.round(2))

# Exporting static partitions to disk
train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)
test_df.to_csv('test_split.csv', index=False)

print("\nPartition files successfully exported: train_split.csv, val_split.csv, test_split.csv")

Class Distribution Comparison Across Splits:


,Train (%),Validation (%),Test (%)
label_idx,,,
0,10.37,10.40,10.40
1,49.09,49.07,49.07
2,40.53,40.53,40.53



Partition files successfully exported: train_split.csv, val_split.csv, test_split.csv
